# Saturday Ops, Day 2: Deliver AI to the Business
**AI Bootcamp | Cortex Agents, Agent Studio, REST API, Managed MCP, CoWork, AI/BI Dashboards**

Day 1 built a governed semantic layer for the **UCLA athletic department**. Today we put it in front of the business:

1. **Recap Day 1** for anyone who missed it (5 minutes).
2. **Build a Cortex Agent in SQL** on top of the semantic view and the search service.
3. **Tour Agent Studio** to see, test and tune the same agent in the UI.
4. **Call the agent over the REST API**.
5. **Expose the agent as a tool in a Snowflake-managed MCP server** so Claude, Cursor or CoCo can use it.
6. **Deliver it in Snowflake CoWork** to each business unit.
7. **Build an AI/BI dashboard** from the same semantic view: one source of truth for AI and BI.
8. *(Optional)* **Ship a Snowflake App Runtime app.**

> **Disclaimer:** All data is synthetic and generated for training purposes. School names are used for illustration only. Coaches and players are generic aliases. Scores, stats, quotes and business figures are fictional.

**Prerequisites:** the `SATURDAY_OPS` database from Day 1, and a role that can create agents and MCP servers (ACCOUNTADMIN or SYSADMIN in a trial account). For section 4 you need a Programmatic Access Token (PAT).

## 0. Setup
Use the same role and warehouse as Day 1. The guard cell checks that Day 1 was run.

In [ ]:
from snowflake.snowpark.context import get_active_session

session = get_active_session()

ROLE = None  # set to the role you used on Day 1 (e.g. "SYSADMIN"); None keeps the current role

for stmt in ([f"USE ROLE {ROLE}"] if ROLE else []) + [
    "CREATE WAREHOUSE IF NOT EXISTS SATURDAY_OPS_WH WAREHOUSE_SIZE = XSMALL AUTO_SUSPEND = 60 AUTO_RESUME = TRUE",
    "USE WAREHOUSE SATURDAY_OPS_WH",
    "CREATE SCHEMA IF NOT EXISTS SATURDAY_OPS.AGENTS",
    "USE SCHEMA SATURDAY_OPS.ANALYTICS",
]:
    session.sql(stmt).collect()

# Guard: Day 1 objects must exist
sv = session.sql("SHOW SEMANTIC VIEWS LIKE 'SATURDAY_OPS_SV' IN SCHEMA SATURDAY_OPS.ANALYTICS").collect()
ss = session.sql("SHOW CORTEX SEARCH SERVICES LIKE 'COACH_PRESSER_SEARCH' IN SCHEMA SATURDAY_OPS.ANALYTICS").collect()
assert sv and ss, "Day 1 objects not found. Run day1/Saturday_Ops_Day1.ipynb first."

print("Ready:", session.sql("SELECT CURRENT_ROLE(), CURRENT_WAREHOUSE()").collect()[0])

## 1. Recap: what we built on Day 1
If you missed Day 1, this is everything Day 2 builds on:

```
SATURDAY_OPS
  RAW         13 synthetic tables: teams, games, team stats, fans, tickets, concessions,
              merch, donations, game recaps, coach press conferences, fan surveys,
              fan service tickets
  ANALYTICS   FAN_INSIGHTS, TICKET_INSIGHTS, GAME_TEXT_INSIGHTS   (Cortex AI Functions)
              TEAM_STYLE_PROFILE   (stats merged with text)
              GAME_BUSINESS_SUMMARY, TEAM_GAME_RESULTS
              SATURDAY_OPS_SV      (semantic view, 10 verified queries)
              COACH_PRESSER_SEARCH (Cortex Search service)
```

The idea: **AI Functions turn text into data, the semantic view gives that data business meaning, and Cortex Search makes the raw text findable.** Today the agent uses all three.

### 1a. The data: structured and unstructured, all generated in SQL

In [ ]:
%%sql -r recap_row_counts
SELECT TABLE_SCHEMA, TABLE_NAME, ROW_COUNT
FROM SATURDAY_OPS.INFORMATION_SCHEMA.TABLES
WHERE TABLE_SCHEMA IN ('RAW', 'ANALYTICS') AND TABLE_TYPE = 'BASE TABLE'
ORDER BY TABLE_SCHEMA DESC, ROW_COUNT DESC

### 1b. Cortex AI Functions turned fan surveys and service tickets into columns

In [ ]:
%%sql -r recap_ai_functions
SELECT t.CATEGORY AS AI_CLASSIFY_CATEGORY, COUNT(*) AS TICKETS,
       ROUND(AVG(f.SENTIMENT_SCORE), 2) AS AVG_AI_SENTIMENT_SAME_GAMES
FROM SATURDAY_OPS.ANALYTICS.TICKET_INSIGHTS t
LEFT JOIN (SELECT GAME_ID, AVG(SENTIMENT_SCORE) AS SENTIMENT_SCORE
           FROM SATURDAY_OPS.ANALYTICS.FAN_INSIGHTS GROUP BY GAME_ID) f ON f.GAME_ID = t.GAME_ID
GROUP BY 1 ORDER BY TICKETS DESC

### 1c. The semantic view: business metrics everyone agrees on
Query it with `SEMANTIC_VIEW()`. The metric definitions live in one place.

In [ ]:
%%sql -r recap_semantic_view
SHOW SEMANTIC METRICS IN SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV

### 1d. Cortex Search: find answers inside the coach press conferences

In [ ]:
%%sql -r recap_search
SELECT r.value:SCHOOL_NAME::STRING AS SCHOOL, r.value:SEASON_WEEK::INT AS WEEK,
       LEFT(r.value:TRANSCRIPT::STRING, 200) AS SNIPPET
FROM TABLE(FLATTEN(PARSE_JSON(SNOWFLAKE.CORTEX.SEARCH_PREVIEW(
  'SATURDAY_OPS.ANALYTICS.COACH_PRESSER_SEARCH',
  '{"query": "we want to run the ball and control the clock", "columns": ["SCHOOL_NAME","SEASON_WEEK","TRANSCRIPT"], "limit": 5}'
)):results)) r

## 2. Build a Cortex Agent in SQL
An agent is a schema-level object. It contains:
- **Models:** `auto`, so Snowflake picks the best orchestration model available in your account.
- **Instructions:** how to respond and how to choose tools.
- **Tools:** Cortex Analyst (structured, through the semantic view), Cortex Search (unstructured, through the presser transcripts), and Data to Chart.
- **Tool resources:** which semantic view, search service and warehouse each tool uses.

The agent doesn't copy any data. It runs with the caller's role, so all existing grants and policies still apply.

In [ ]:
%%sql -r create_agent
CREATE OR REPLACE AGENT SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT
  COMMENT = 'Saturday Ops game-day assistant for UCLA Athletics. Synthetic data.'
  PROFILE = '{"display_name": "Saturday Ops Assistant", "color": "blue"}'
  FROM SPECIFICATION
$$
models:
  orchestration: auto

orchestration:
  budget:
    seconds: 60
    tokens: 32000

instructions:
  response: >
    You are the Saturday Ops assistant for the UCLA athletic department. Answer concisely for
    business users in Ticketing, Concessions and Merch, Development (donors), Fan Experience and
    Football Ops. Lead with the number or finding, then one or two sentences of context.
    Format currency as dollars and rates as percentages. When you quote a coach, name the school and week.
  orchestration: >
    Use Saturday_Ops_Analyst for any metric, count, revenue, attendance, donation, sentiment, complaint
    or play-style statistic. Use Coach_Presser_Search for what coaches said in press conferences
    (quotes, stated game plans, injuries, tone). For questions that compare what coaches say with what
    teams actually do, call BOTH tools and combine the results. Use data_to_chart when the user asks for
    a chart or when a trend over weeks is returned.
  sample_questions:
    - question: "Which coaches say they are run-first but actually pass a lot? Show me what they said."
    - question: "Do donations go up after wins compared to losses?"
    - question: "What were the top complaint categories at the Miami game?"
    - question: "Chart average fan sentiment by home game week."

tools:
  - tool_spec:
      type: "cortex_analyst_text_to_sql"
      name: "Saturday_Ops_Analyst"
      description: >
        Answers questions over structured Saturday Ops data through the SATURDAY_OPS_SV semantic view:
        UCLA home game ticket, concession and merch revenue, attendance and fill rate, donations after games,
        fan sentiment and service complaint categories, and conference-wide team play style and pass rates.
        Do NOT use for press conference quotes.
  - tool_spec:
      type: "cortex_search"
      name: "Coach_Presser_Search"
      description: >
        Searches post-game coach press conference transcripts for all 12 conference teams. Filterable by
        SCHOOL_NAME, SEASON_WEEK, RESULT (W/L) and MEDIA_STYLE_LABEL. Use for quotes and stated game plans.
        Do NOT use for numbers.
  - tool_spec:
      type: "data_to_chart"
      name: "data_to_chart"
      description: "Generates visualizations from data returned by the analyst tool."

tool_resources:
  Saturday_Ops_Analyst:
    semantic_view: "SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV"
    execution_environment:
      type: warehouse
      warehouse: "SATURDAY_OPS_WH"
      query_timeout: 60
  Coach_Presser_Search:
    search_service: "SATURDAY_OPS.ANALYTICS.COACH_PRESSER_SEARCH"
    id_column: "PRESSER_ID"
    title_column: "SCHOOL_NAME"
    max_results: 5
$$

In [ ]:
%%sql -r describe_agent
DESCRIBE AGENT SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT

### Test the agent from SQL
`SNOWFLAKE.CORTEX.DATA_AGENT_RUN` returns the full response as JSON, including the tools the agent called, the SQL it wrote and its final answer. The small helper below prints the useful parts.

In [ ]:
import json

AGENT = "SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT"

def ask(question: str, show_trace: bool = True) -> dict:
    body = json.dumps({"messages": [{"role": "user", "content": [{"type": "text", "text": question}]}]})
    raw = session.sql("SELECT SNOWFLAKE.CORTEX.DATA_AGENT_RUN(?, ?)", params=[AGENT, body]).collect()[0][0]
    resp = json.loads(raw)
    print(f"Q: {question}\n")
    for item in resp.get("content", []):
        kind = item.get("type")
        if show_trace and kind == "tool_use":
            print(f"  [tool] {item['tool_use'].get('name')}")
        elif show_trace and kind == "tool_result":
            for c in item["tool_result"].get("content", []):
                sql = c.get("json", {}).get("sql") if isinstance(c.get("json"), dict) else None
                if sql:
                    print("  [sql]", " ".join(sql.split())[:300])
        elif kind == "text":
            print("\nA:", item["text"])
    return resp

_ = ask("Do donations go up after wins compared to losses?")

In [ ]:
# The Day 1 payoff question, in plain English. The agent should call BOTH tools.
_ = ask("Which coaches say they are run-first but actually pass a lot? Quote what they said.")

## 3. Agent Studio: the same agent, in the UI
Everything you just did in SQL is visible and editable in Snowsight. SQL, REST and the UI all edit **one agent object**, so you can build in SQL for CI/CD and Git and tune in the UI with business users.

1. Go to **AI & ML > Agents** and open **Saturday Ops Assistant** (`SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT`).
2. **About:** display name, description and the sample questions from the spec.
3. **Tools:** `Saturday_Ops_Analyst` (semantic view), `Coach_Presser_Search` (search service), `data_to_chart`. Click a tool to see its description, which is the most important prompt you write.
4. **Orchestration / Instructions:** the response and routing instructions. Try editing the response tone and click **Save**, then rerun `DESCRIBE AGENT` above to see that the change landed in the same object.
5. **Playground (right pane):** ask *"Chart average fan sentiment by home game week"*. Expand the **thinking / trace** to see the planning, the tool calls, the generated SQL and the chart.
6. **Access:** see which roles can use the agent. We grant it in section 6.
7. **Monitoring / Evaluations:** every run is traced (latency, tools, tokens, user feedback). Use evaluations to score the agent against your verified questions before you roll it out.

## 4. Query the agent over the REST API (from your laptop)
Any application (a web app, Slack bot, Teams, or a Python script) can call the agent:

`POST https://<account_url>/api/v2/databases/SATURDAY_OPS/schemas/AGENTS/agents/SATURDAY_OPS_AGENT:run`

**Step 1: create a Programmatic Access Token (PAT).** In Snowsight, go to your user menu > **Settings > Authentication > Programmatic access tokens > Generate new token**. Restrict it to the role you used today. Or use SQL (see the next cell). Copy the token once; it is shown only once.

**Step 2: find your account URL.** The next cell prints it.

**Step 3: run the script below on your laptop** (`pip install requests`). It is a plain HTTPS call, so you need nothing from Snowflake installed.

In [ ]:
%%sql -r account_url
-- Your REST base URL. Use hyphens, not underscores, in the host name.
SELECT 'https://' || LOWER(REPLACE(CURRENT_ORGANIZATION_NAME() || '-' || CURRENT_ACCOUNT_NAME(), '_', '-'))
       || '.snowflakecomputing.com' AS ACCOUNT_URL,
       CURRENT_USER() AS USER_NAME, CURRENT_ROLE() AS ROLE_NAME

-- Optional: create a PAT in SQL instead of the UI (copy TOKEN_SECRET from the result):
-- ALTER USER ADD PROGRAMMATIC ACCESS TOKEN SATURDAY_OPS_PAT ROLE_RESTRICTION = 'SYSADMIN' DAYS_TO_EXPIRY = 7;

**`ask_saturday_ops.py`** (run on your laptop, not in this notebook):

```python
import os, json, requests

ACCOUNT_URL = "https://<org>-<account>.snowflakecomputing.com"   # from the cell above
PAT = os.environ["SNOWFLAKE_PAT"]                                  # export SNOWFLAKE_PAT=...

url = f"{ACCOUNT_URL}/api/v2/databases/SATURDAY_OPS/schemas/AGENTS/agents/SATURDAY_OPS_AGENT:run"
headers = {
    "Authorization": f"Bearer {PAT}",
    "Content-Type": "application/json",
    "Accept": "application/json",
    "X-Snowflake-Role": "SYSADMIN",
}
body = {
    "stream": False,
    "messages": [{"role": "user", "content": [
        {"type": "text", "text": "What were the top complaint categories at the Miami game?"}]}],
}

resp = requests.post(url, headers=headers, json=body, timeout=300)
resp.raise_for_status()
for item in resp.json().get("content", []):
    if item["type"] == "tool_use":
        print("[tool]", item["tool_use"]["name"])
    elif item["type"] == "text":
        print(item["text"])
```

**Streaming version** (what a chat UI uses). Set `"stream": True` and `Accept: text/event-stream`, then read the server-sent events:

```python
with requests.post(url, headers={**headers, "Accept": "text/event-stream"},
                   json={**body, "stream": True}, stream=True, timeout=300) as r:
    event = None
    for line in r.iter_lines(decode_unicode=True):
        if line.startswith("event:"):
            event = line[6:].strip()
        elif line.startswith("data:") and event == "response.text.delta":
            print(json.loads(line[5:])["text"], end="", flush=True)
```

**curl equivalent:**
```bash
curl -X POST "$ACCOUNT_URL/api/v2/databases/SATURDAY_OPS/schemas/AGENTS/agents/SATURDAY_OPS_AGENT:run" \
  -H "Authorization: Bearer $SNOWFLAKE_PAT" -H "Content-Type: application/json" -H "Accept: application/json" \
  -d '{"stream": false, "messages":[{"role":"user","content":[{"type":"text","text":"Do donations go up after wins?"}]}]}'
```

The answer matches what you get in Snowsight, because it comes from the same agent, semantic view and grants.

## 5. Add the agent as a tool in a Snowflake-managed MCP server
**MCP (Model Context Protocol)** is the open standard AI clients (Claude Desktop, Cursor, CoCo, custom agents) use to discover and call tools. A **Snowflake-managed MCP server** is a schema object that publishes Snowflake objects as MCP tools. It runs inside Snowflake, so there's nothing to host and every call goes through your RBAC.

We expose three tools:
- `saturday-ops-agent`: the **whole agent** (`CORTEX_AGENT_RUN`)
- `saturday-ops-analyst`: the semantic view directly (`CORTEX_ANALYST_MESSAGE`)
- `coach-presser-search`: the search service directly (`CORTEX_SEARCH_SERVICE_QUERY`)

In [ ]:
%%sql -r create_mcp_server
CREATE OR REPLACE MCP SERVER SATURDAY_OPS.AGENTS.SATURDAY_OPS_MCP
  FROM SPECIFICATION
$$
tools:
  - name: "saturday-ops-agent"
    type: "CORTEX_AGENT_RUN"
    identifier: "SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT"
    title: "Saturday Ops Assistant"
    description: "Ask any UCLA game-day business or conference play-style question. Combines metrics and coach press conference quotes."
    annotations:
      readOnlyHint: true
      destructiveHint: false
      idempotentHint: true
      openWorldHint: false
  - name: "saturday-ops-analyst"
    type: "CORTEX_ANALYST_MESSAGE"
    identifier: "SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV"
    title: "Saturday Ops metrics"
    description: "Text-to-SQL over the Saturday Ops semantic view: revenue, attendance, donations, sentiment, complaints, play style."
  - name: "coach-presser-search"
    type: "CORTEX_SEARCH_SERVICE_QUERY"
    identifier: "SATURDAY_OPS.ANALYTICS.COACH_PRESSER_SEARCH"
    title: "Coach press conference search"
    description: "Semantic search over coach press conference transcripts for all 12 conference teams."
$$

In [ ]:
%%sql -r describe_mcp_server
DESCRIBE MCP SERVER SATURDAY_OPS.AGENTS.SATURDAY_OPS_MCP

### Connect a client to the MCP server
**Endpoint** (use the `ACCOUNT_URL` from section 4):

`https://<org>-<account>.snowflakecomputing.com/api/v2/databases/SATURDAY_OPS/schemas/AGENTS/mcp-servers/SATURDAY_OPS_MCP`

**Claude Desktop / Cursor (`mcp.json`)**, authenticated with the same PAT:
```json
{
  "mcpServers": {
    "saturday-ops": {
      "url": "https://<org>-<account>.snowflakecomputing.com/api/v2/databases/SATURDAY_OPS/schemas/AGENTS/mcp-servers/SATURDAY_OPS_MCP",
      "headers": { "Authorization": "Bearer <PAT>" }
    }
  }
}
```

**Raw JSON-RPC from your laptop** (shows what any MCP client does under the hood):
```python
import os, requests
MCP = f"{ACCOUNT_URL}/api/v2/databases/SATURDAY_OPS/schemas/AGENTS/mcp-servers/SATURDAY_OPS_MCP"
H = {"Authorization": f"Bearer {os.environ['SNOWFLAKE_PAT']}", "Content-Type": "application/json"}

# 1. Discover tools
print(requests.post(MCP, headers=H, json={"jsonrpc": "2.0", "id": 1, "method": "tools/list", "params": {}}).json())

# 2. Call the agent tool
print(requests.post(MCP, headers=H, json={
    "jsonrpc": "2.0", "id": 2, "method": "tools/call",
    "params": {"name": "saturday-ops-agent",
               "arguments": {"message": "Does attendance go up against air raid teams?"}}}).json())
```

**Try it:** in CoCo or Claude Desktop with the server connected, ask *"Using the Saturday Ops tools, which opponent style draws the best attendance, and what did that coach say after the game?"* and watch the client pick the tool.

## 6. Deliver the agent in Snowflake CoWork
CoWork is the business-user front door: a chat app where people talk to the agents they have access to. No SQL, no notebooks.

**Step 1: grant access.** Grant the agent plus the objects behind it to a business role. The agent runs as the caller, so the caller needs the underlying access too. The cell below grants it to `PUBLIC` for the bootcamp. In production, use a role for each business unit.

In [ ]:
%%sql -r grant_cowork
-- Bootcamp only: grant to PUBLIC. In production, use per-business-unit roles.
EXECUTE IMMEDIATE $$
BEGIN
  GRANT USAGE ON DATABASE SATURDAY_OPS TO ROLE PUBLIC;
  GRANT USAGE ON SCHEMA SATURDAY_OPS.ANALYTICS TO ROLE PUBLIC;
  GRANT USAGE ON SCHEMA SATURDAY_OPS.AGENTS TO ROLE PUBLIC;
  GRANT USAGE ON AGENT SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT TO ROLE PUBLIC;
  GRANT SELECT ON SEMANTIC VIEW SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV TO ROLE PUBLIC;
  GRANT SELECT ON ALL TABLES IN SCHEMA SATURDAY_OPS.ANALYTICS TO ROLE PUBLIC;
  GRANT SELECT ON ALL VIEWS IN SCHEMA SATURDAY_OPS.ANALYTICS TO ROLE PUBLIC;
  GRANT USAGE ON CORTEX SEARCH SERVICE SATURDAY_OPS.ANALYTICS.COACH_PRESSER_SEARCH TO ROLE PUBLIC;
  GRANT USAGE ON WAREHOUSE SATURDAY_OPS_WH TO ROLE PUBLIC;
  RETURN 'Granted';
END;
$$

**Step 2: open CoWork.** In Snowsight, go to **AI & ML > Snowflake CoWork** (or the CoWork URL for your account) and select **Saturday Ops Assistant** in the agent picker. If you don't see it, an admin may need to add it to the account's CoWork agent list.

**Step 3: try one question per business unit:**
| Business unit | Ask CoWork |
|---|---|
| Ticketing | *How does revenue per attendee at the USC rivalry game compare to other home games?* |
| Concessions and Merch | *Total game-day revenue by kickoff window. Chart it.* |
| Development | *Do donations go up after wins compared to losses?* |
| Fan Experience | *What were the top complaint categories at the Miami game, and how was fan sentiment that week?* |
| Football Ops | *Which coaches say they're run-first but actually pass a lot? Show me the quotes.* |

What to look for: the **thinking trace**, the **generated SQL** and its **verified query** badge, **citations** that link to the press conference, **follow-up questions** that keep context, the **chart**, and **thumbs up/down** feedback (which appears in Agent Studio monitoring).

Each business unit asks different questions, but every number comes from the same governed semantic view.

## 7. AI/BI dashboards on the same semantic view
The agent answers questions on demand. Dashboards answer the questions leaders ask every Monday. Because both are grounded in `SATURDAY_OPS_SV`, a metric like *revenue per attendee* means the same thing in both.

**Build it with AI in Snowsight:**
1. Ask CoCo in Workspaces: *"Build a dashboard from SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV"*.
2. Choose **Generate with AI** and pick the semantic view as the source.
3. Prompt: *"Game-day executive dashboard for UCLA Athletics: total revenue by home game, revenue per attendee, fill rate vs opponent style, donations after wins vs losses, fan sentiment by week, and top complaint categories. Add an opponent filter."*
4. Review the generated tiles. Each one is a semantic view query, so there's no hand-written SQL to drift from the metric definitions.
5. Edit a tile in plain English

The cell below is the SQL behind a typical tile. It queries the semantic view by metric name, not by raw columns.

In [ ]:
%%sql -r dashboard_tile_example
-- A typical dashboard tile: metrics by name from the semantic view, no hand-written joins.
SELECT * FROM SEMANTIC_VIEW(
  SATURDAY_OPS.ANALYTICS.SATURDAY_OPS_SV
  DIMENSIONS home_games.home_week, home_games.home_opponent, home_games.opponent_style
  METRICS home_games.total_revenue, home_games.per_cap_spend, home_games.attendance_pct
)
ORDER BY home_week

### Keep the semantic layer the source of truth
Change a definition once in `SATURDAY_OPS_SV` (for example, add a synonym or a new metric such as *per cap concessions*) and the change appears in **Cortex Analyst, the agent, CoWork, the MCP server, the dashboard and the app**. Nobody updates six tools by hand.

## 8. Snowflake App Runtime: Game Day Command Center
Some teams want a branded experience instead of a chat window. **Game Day Command Center** (`day2/game-day-command-center/`) is a Next.js web app that runs **inside Snowflake** as an `APPLICATION SERVICE` (`SATURDAY_OPS.AGENTS.GAME_DAY_COMMAND_CENTER`). It has three tabs:

| Tab | What it shows | Source |
|---|---|---|
| Game Day Overview | Revenue, per cap, fill rate, post-game giving, revenue by game, scorecard | `SEMANTIC_VIEW(SATURDAY_OPS_SV ...)` |
| Fan Experience | AI sentiment by week, AI-classified tickets by category | `SEMANTIC_VIEW(SATURDAY_OPS_SV ...)` |
| **Ask the Agent** | Chat with `SATURDAY_OPS_AGENT`, with a "How I got this" panel showing the tools used and the SQL | `DATA_AGENT_RUN` |

The agent tab's server route (`app/api/agent/route.ts`) runs `SNOWFLAKE.CORTEX.DATA_AGENT_RUN('SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT', ?)` with the conversation passed as a bind parameter. It's the same agent as Agent Studio, REST, MCP and CoWork, with no PAT and no external access, because the app already runs in Snowflake.

**Deploy it** from a terminal with the Snowflake CLI (3.26+), using the role you used today:
```bash
cd day2/game-day-command-center
snow app deploy        # upload, then remote build, then create the APPLICATION SERVICE; prints the URL
```
The next cell finds the app and its URL.

In [ ]:
%%sql -r find_app
SHOW APPLICATION SERVICES IN SCHEMA SATURDAY_OPS.AGENTS

Open the URL from `DESCRIBE APPLICATION SERVICE SATURDAY_OPS.AGENTS.GAME_DAY_COMMAND_CENTER` (or the one `snow app deploy` printed), then go to **Ask the Agent** and click a sample question.

**Share it with a business role** (the app runs queries with its owner's rights, so viewers only need access to the app):
```sql
GRANT USAGE ON APPLICATION SERVICE SATURDAY_OPS.AGENTS.GAME_DAY_COMMAND_CENTER TO ROLE <business_role>;
```

> Talking point: *"Same agent, same semantic view, now in a custom app with your branding, running inside the Snowflake security boundary."*

### What you built today
- 1 Cortex Agent (`SATURDAY_OPS_AGENT`) with Analyst, Search and Chart tools, built in SQL
- The same agent in **Agent Studio**, over the **REST API**, through a **managed MCP server** (`SATURDAY_OPS_MCP`) and in **CoWork**
- An **AI/BI dashboard** grounded in the same semantic view
- A **Snowflake App Runtime app** (`GAME_DAY_COMMAND_CENTER`) with KPI tabs and an **Ask the Agent** tab
- One semantic layer that every surface uses

## 9. Cleanup (optional)
Drops only the Day 2 objects. Day 1 data stays.

In [ ]:
# Uncomment to remove Day 2 objects
# session.sql("DROP APPLICATION SERVICE IF EXISTS SATURDAY_OPS.AGENTS.GAME_DAY_COMMAND_CENTER").collect()
# session.sql("DROP MCP SERVER IF EXISTS SATURDAY_OPS.AGENTS.SATURDAY_OPS_MCP").collect()
# session.sql("DROP AGENT IF EXISTS SATURDAY_OPS.AGENTS.SATURDAY_OPS_AGENT").collect()
# session.sql("DROP SCHEMA IF EXISTS SATURDAY_OPS.AGENTS").collect()
# session.sql("ALTER USER REMOVE PROGRAMMATIC ACCESS TOKEN SATURDAY_OPS_PAT").collect()